# Chapter 4 — Deep Dive: State-of-the-Art AI EDA Solutions

**Multi-Agent Analog & Digital EDA — PhD-Level Monograph (Notebook Form)**

---

This chapter dissects representative *agentic* and *LLM-centric* EDA stacks that have appeared in the 2023–2026 research arc. We treat each system as a **stochastic policy** over design artifacts (netlists, RTL, scripts, constraints) coupled to **verifiable oracles** (simulators, linters, formal engines). Where original papers rely on proprietary weights or cloud APIs, we provide **self-contained simulators** that preserve the *control-flow structure* and *learning objectives*.

### Learning objectives

1. **Decompose** ChatEDA-style controllers into *task decomposition → script synthesis → guarded execution*.
2. **Formalize** RTLCoder-like *quality-based reinforcement* from compiler/simulator feedback.
3. **Analyze** EDAid-style *divergent CoT ensembles* as variance-reduction in planning.
4. **Model** ACE-RTL *context evolution* as iterated Bayesian / information-gain updates on a latent design state.
5. **Compare** systems on interpretable axes using quantitative **radar charts** and **performance tables**.

### Mathematical notation (used throughout)

- Design requirement $r \in \mathcal{R}$ (natural language + structured specs).
- Latent decomposition $z = (z_1,\ldots,z_k)$, $z_i \in \mathcal{Z}$ (subgoals).
- Executable script $a \in \mathcal{A}$ (Python/Tcl/YAML fragments).
- Environment transition $s' = \mathcal{T}(s,a)$ with reward or score $Q(s,a)\in\mathbb{R}$.
- LLM policy $\pi_\theta(a\mid s,r)$; fine-tuned variant $\pi_{\theta}^{\text{AutoMage}}$, etc.

---


## 4.0 Taxonomy: Where Each System Sits

| System | Primary artifact | Verification hook | Multi-agent? | On-prem friendly? |
|--------|------------------|-------------------|--------------|-------------------|
| **ChatEDA / AutoMage** | Tooling scripts (Python/Tcl) | EDA tool stdout / exit codes | Controller + tools | Partial (model size) |
| **RTLCoder** | RTL (Verilog/SystemVerilog) | **Icarus Verilog** (open) | Single model + loop | **Strong** (7B / 4GB) |
| **EDAid** | Plans + edits | Simulation / lint (conceptual) | **Yes** (diverse CoT) | Depends on backbone |
| **ACE-RTL** | RTL + evolving context | Iterative regression suite | Agentic context memory | Depends on deployment |

**Key idea:** modern EDA LLMs are not monolithic chatbots; they are **closed-loop controllers** $\mathcal{C}: (r,s_t) \mapsto a_t$ with *typed actions* and *typed observations*.

---


### 4.0.1 Self-instruction and teacher–student distillation (AutoMage context)

A practical recipe for building $\pi_{\theta}^{\text{AutoMage}}$ proceeds in three waves:

1. **Seed collection:** human or weak-model annotations on a small set of EDA tasks $\{r_j\}_{j=1}^{J}$.
2. **Teacher expansion:** a high-capacity model $T$ (e.g., GPT-4-class) generates diverse decompositions and scripts:
   $$
   (z_j^{(\ell)}, a_j^{(\ell)}) \sim T(\cdot \mid r_j), \quad \ell = 1,\ldots,L_j.
   $$
3. **Filtering & SFT:** tuples are filtered by static checks / sandbox dry-runs, yielding $\mathcal{D}$ with $|\mathcal{D}| \approx 10^4$.

The student minimizes standard causal LM loss with **tool-aware formatting** (e.g., XML-tagged stages) so that inference-time parsers can route stages deterministically. Let $x = [r; z]$ be the concatenated prompt; then SFT is
$$
\min_\theta \; - \sum_{i} \sum_{t} \log p_\theta(a_t^{(i)} \mid x^{(i)}, a_{<t}^{(i)}).
$$

**Takeaway:** AutoMage is not merely “Llama + EDA PDFs”; it is a *scheduling head* trained on structured supervision that mirrors the three-stage runtime pipeline.

---


In [ ]:
# Global imports & visualization defaults (self-contained; no API keys)
from __future__ import annotations

import json
import math
import random
import re
import textwrap
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Tuple

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

# Reproducibility
RNG = np.random.default_rng(42)
random.seed(42)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.alpha": 0.6,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)
pio.templates.default = "plotly_dark"

print("matplotlib + plotly dark theme configured (facecolor", DARK_BG + ").")


## 4.1 ChatEDA and the **AutoMage** Controller

### 4.1.1 Operational semantics: a three-stage pipeline

ChatEDA-style systems instantiate a **hierarchical policy**:

1. **Task decomposition** maps a requirement $r$ to an ordered (or parallel) set of subgoals:
   $$
   q_\phi(z \mid r) \approx \text{Decomposer}(r), \quad z = (z_1,\ldots,z_k).
   $$
2. **Script generation** conditions on $(r,z)$ and emits tool-facing code $a$ in domain-specific languages (often **Python** orchestration + **Tcl** for legacy tools):
   $$
   \pi_\theta(a \mid r, z) \approx \text{Synthesizer}(r,z).
   $$
3. **Task execution** applies $a$ in a sandboxed EDA environment, yielding observation $o$ and next state $s'$:
   $$
   (s', o) = \mathcal{E}(s,a), \quad r_t = \text{Reward}(o).
   $$

**AutoMage** refers to a controller LLM fine-tuned (e.g., from **Llama 2**) using a **self-instruction** loop: seed tasks are expanded into diverse supervision, then distilled into the student. A common construction (as reported in the literature) uses a stronger teacher (e.g., **GPT-4**) to produce on the order of **$\sim 10^4$** tuples:
$$
\mathcal{D} = \{(r^{(i)}, z^{(i)}, a^{(i)})\}_{i=1}^{N}, \quad N \approx 10{,}000,
$$
followed by supervised fine-tuning (SFT) minimizing
$$
\mathcal{L}_{\text{SFT}}(\theta) = - \mathbb{E}_{(r,z,a)\sim\mathcal{D}} \big[ \log \pi_\theta(a \mid r, z) \big].
$$

### 4.1.2 Why three stages matter for EDA

- **Compositionality:** EDA flows are *long-horizon*; explicit decomposition reduces **credit assignment error** when a downstream tool fails.
- **Safety:** Generated scripts can be **statically scanned** before execution (allow-lists, path constraints).
- **Auditability:** $(r,z,a,o)$ traces support regression on regressions—critical for industrial sign-off.

---


In [ ]:
# --- Simplified ChatEDA / AutoMage pipeline simulator (mock data) ---

@dataclass
class ChatEDATrace:
    requirement: str
    decomposition: List[str]
    script_python: str
    script_tcl: str
    stdout: str
    success: bool
    tool_reward: float


def mock_decompose(requirement: str, k: int = 4) -> List[str]:
    '''Template decomposition — stand-in for q_phi(z|r).'''
    req_l = requirement.lower()
    steps = []
    if "synthesis" in req_l or "dc" in req_l:
        steps += ["Load libraries & MMMC views", "Read RTL & elaborate", "Set constraints (SDC)", "Run compile_ultra"]
    elif "apr" in req_l or "place" in req_l:
        steps += ["Floorplan IO & macros", "Power grid generation", "Placement", "CTS & post-route optimization"]
    else:
        steps += ["Parse inputs", "Validate technology files", "Run main EDA step", "Generate reports"]
    while len(steps) < k:
        steps.append(f"Secondary verification {len(steps)+1}")
    return steps[:k]


def mock_scriptgen(req: str, decomp: List[str]) -> Tuple[str, str]:
    '''Emit plausible Python + Tcl — stand-in for pi_theta(a|r,z).'''
    py = textwrap.dedent(f'''
    # Auto-generated orchestration (mock)
    import subprocess, pathlib
    WORK = pathlib.Path("./work_chat_eda_mock").resolve()
    WORK.mkdir(exist_ok=True)
    steps = {json.dumps(decomp)}
    for i, s in enumerate(steps, 1):
        print(f"[PY] Step {{i}}: {{s}}")
    ''').strip()

    tcl = "\n".join([
        "# Mock Synopsys-style Tcl fragment",
        "set_db / .lib_setup_done true",
        "foreach step {" + " ".join(f"{{{d}}}" for d in decomp) + "} {",
        "  puts \"[TCL] running: $step\"",
        "}",
    ])
    return py, tcl


def mock_execute(py: str, tcl: str) -> Tuple[str, bool, float]:
    '''Toy environment — checks structural properties, not real tools.'''
    ok_py = "subprocess" in py and "steps" in py
    ok_tcl = "foreach" in tcl and "puts" in tcl
    success = ok_py and ok_tcl
    # Synthetic reward: completeness minus a noise term
    reward = 1.0 if success else 0.35
    reward -= 0.05 * RNG.normal()
    stdout = "[ENV] executed mock pipeline\n" + ("STATUS=PASS\n" if success else "STATUS=WARN\n")
    return stdout, success, float(np.clip(reward, 0, 1))


def run_chat_eda(requirement: str) -> ChatEDATrace:
    z = mock_decompose(requirement)
    py, tcl = mock_scriptgen(requirement, z)
    out, ok, rw = mock_execute(py, tcl)
    return ChatEDATrace(requirement, z, py, tcl, out, ok, rw)


req_samples = [
    "Run DC synthesis for the RISC-V core with 500 MHz target and UPF-aware power intent.",
    "Perform ICC2 APR for a 12nm FinFET testchip with dual power domains.",
]

traces = [run_chat_eda(r) for r in req_samples]
for tr in traces:
    print("=" * 72)
    print("Requirement:", tr.requirement[:88], "...")
    print("Decomposition:", tr.decomposition)
    print("Reward:", round(tr.tool_reward, 3), "| success:", tr.success)
    print("--- Tcl excerpt ---\n", "\n".join(tr.script_tcl.splitlines()[:6]), "\n...")

# Visual: pipeline as a block diagram
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_facecolor(DARK_BG)
fig.patch.set_facecolor(DARK_BG)
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")
boxes = [
    (0.3, 0.8, 2.4, 1.4, "Decomposer\n$q_\\phi(z|r)$", "#3fb950"),
    (4.0, 0.8, 2.6, 1.4, "AutoMage\n$\\pi_\\theta(a|r,z)$", ACCENT),
    (7.8, 0.8, 2.4, 1.4, "EDA Env\n$\\mathcal{E}(s,a)$", "#d29922"),
]
for x, y, w, h, label, c in boxes:
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                facecolor=c, alpha=0.18, edgecolor=c, linewidth=2))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", color=c, fontsize=10, fontweight="bold")
for (x1, w1), (x2, _) in zip([(b[0], b[2]) for b in boxes], [(b[0], b[2]) for b in boxes][1:]):
    ax.annotate("", xy=(x2 - 0.05, 1.5), xytext=(x1 + w1 + 0.05, 1.5),
                arrowprops=dict(arrowstyle="->", color="#8b949e", lw=1.8))
ax.text(6, 2.55, "ChatEDA-style closed loop (conceptual)", ha="center", color="#c9d1d9", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 4.2 RTLCoder: Lightweight, Verifiable RTL Synthesis

### 4.2.1 Deployment profile

**RTLCoder** emphasizes a **7B-class** decoder-only transformer (e.g., **DeepSeek**-family or **Mistral**-family derivatives) that can run on **$\sim$4GB** VRAM with aggressive quantization (INT4/INT8) and KV-cache trimming—enabling **on-prem** inference.

### 4.2.2 IP privacy argument

Let $X_{\text{IP}}$ denote proprietary RTL. Cloud APIs incur an information leakage channel modeled loosely as mutual information $I(X_{\text{IP}}; \text{Cloud}) > 0$ under logging, fine-tuning, or prompt retention policies. Local inference enforces
$$
I(X_{\text{IP}}; \text{Cloud}) \approx 0
$$
under an air-gapped threat model (still subject to side channels, supply chain, etc., but strictly superior for IP-centric organizations).

### 4.2.3 Quality-Based Training (QBT) with Icarus Verilog

RTLCoder-style training uses **compiler/simulator feedback** as a *cheap verifier*. Let $V(a)$ be a vector of diagnostics (syntax OK, elaboration OK, pass/fail of a tiny testbench). Define a scalar quality
$$
Q(a) = w^\top \phi(V(a)), \quad \phi \text{ a feature map.}
$$

A pragmatic surrogate objective blends maximum likelihood on good samples and **down-weights** bad ones:
$$
\mathcal{L}_{\text{QBT}}(\theta)
= - \mathbb{E}_{a \sim \pi_\theta(\cdot \mid s)} \Big[ \sigma\big(Q(a)\big) \log \pi_\theta(a \mid s) \Big],
$$
with $\sigma$ a squashing nonlinearity (e.g., logistic). This is related to **RLHF** / **RLAIF**, but with a *deterministic*, open-source oracle (Icarus) instead of human preference models.

### 4.2.4 Memory budget sketch (7B on $\sim$4GB)

Let $P$ be parameter count, $b$ bits per weight after quantization, and assume a simplified peak memory for inference
$$
M_{\text{W}} \approx \frac{P \cdot b}{8} \ \text{bytes}, \quad P \approx 7\times 10^9.
$$
With $b=4$ (INT4), $M_{\text{W}} \approx 3.5\,$GB before activations, KV-cache, and runtime overhead—hence the engineering claim “fits in 4GB” requires **weight-only quantization**, **short context**, and often **CPU offload** for attention buffers. Training loops that keep optimizer states resident are *not* assumed here; QBT is typically applied in data-center phases, while deployment stays edge-friendly.

---


In [ ]:
# --- RTLCoder-style training loop (synthetic quality from a mock Verilog linter) ---

def mock_iv_features(rtl: str) -> Dict[str, float]:
    '''Pretend Icarus Verilog / parser feedback.'''
    feats = {
        "syntax_ok": float(bool(re.search(r"\bmodule\b.*\bendmodule\b", rtl, re.S))),
        "has_timescale": float("`timescale" in rtl),
        "balanced_ports": float(rtl.count("input ") + rtl.count("output ") > 0),
    }
    # toy test: always block sensitivity
    feats["combo_sensitivity"] = float(bool(re.search(r"always\s*@\s*\*", rtl)))
    return feats


def quality_score(feats: Dict[str, float]) -> float:
    w = np.array([1.2, 0.4, 0.5, 0.3])
    x = np.array([feats[k] for k in ["syntax_ok", "has_timescale", "balanced_ports", "combo_sensitivity"]])
    z = float(w @ x + 0.05 * RNG.normal())
    return float(1 / (1 + math.exp(-z)))  # logistic squashing -> (0,1)


def generate_rtl_candidate(step: int) -> str:
    '''Synthetic policy output improving over training steps (mock curriculum).'''
    base = "`timescale 1ns/1ps\nmodule adder #(parameter W=8)(input [W-1:0] a,b, output [W:0] y);\n"
    if step < 5:
        return base + "endmodule\n"  # incomplete body -> low Q
    if step < 12:
        return base + "  assign y = a + b;\nendmodule\n"  # combinational OK
    return base + "  always @(*) y = a + b;\nendmodule\n"  # still OK; different style


history: List[Tuple[int, float, Dict[str, float]]] = []
for t in range(20):
    rtl = generate_rtl_candidate(t)
    feats = mock_iv_features(rtl)
    Q = quality_score(feats)
    history.append((t, Q, feats))

df_rtl = pd.DataFrame([{"step": h[0], "Q": h[1], **h[2]} for h in history])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df_rtl["step"], df_rtl["Q"], color=ACCENT, lw=2, marker="o", ms=4)
ax.set_xlabel("Mock fine-tuning step $t$")
ax.set_ylabel("Quality $Q(a)$")
ax.set_title("Synthetic QBT curve (Icarus-style features)")
ax.grid(True)
plt.tight_layout()
plt.show()

display(df_rtl.tail(6))


## 4.3 EDAid: Divergent Thought Collaboration

### 4.3.1 Ensemble of Chain-of-Thought (CoT) specialists

EDAid posits $M$ agents $\{\mathcal{A}_m\}_{m=1}^M$, each instantiated from a backbone such as **ChipLlama** but with **different few-shot CoT prompts** $P_m$. Agent $m$ produces a reasoning trace $c_m$ and plan $\pi_m$:
$$
(c_m, \pi_m) \sim \eta_{\psi_m}(\cdot \mid r), \quad \psi_m = (\text{weights}, P_m).
$$

### 4.3.2 Why divergence helps (a variance argument)

Let the true optimal plan be $\pi^\star$. Single-agent greedy decoding can collapse onto **high-probability but wrong** reasoning paths when the posterior is multi-modal. Define group disagreement
$$
\Delta = \frac{1}{M}\sum_m d(\pi_m, \bar\pi), \quad \bar\pi = \frac{1}{M}\sum_m \pi_m \text{ (embedding space proxy)}.
$$

Empirically, moderate $\Delta$ correlates with **lower collision rate** on systematic errors: if all agents share the same flawed heuristic, errors correlate; **diverse CoTs decohere** those failures.

### 4.3.3 Selection / fusion

A meta-controller aggregates via **self-consistency**, **majority vote** on discrete decisions, or **learned fusion**:
$$
\hat\pi = \arg\max_{\pi \in \{\pi_m\}} \sum_{m=1}^M \mathbf{1}[V(\pi_m) \approx V(\pi)] \cdot \text{score}_m.
$$

### 4.3.4 ChipLlama backbone (conceptual)

**ChipLlama** denotes LLMs continued-pretrained on RTL/GLS/netlist-adjacent corpora. Each EDAid agent can be written as the same base $\text{ChipLlama}$ with **different adapter/prompt parameters** $\{P_m\}$ only—mirroring multi-head expert routing without full weight replication.

### 4.3.5 Toy collision model: why divergence lowers systemic failure rate

Suppose each agent independently commits a catastrophic planning error with probability $p$, and errors are *identical* (fully correlated) when prompts are the same. Then the ensemble fails if all agree on the bad plan: failure probability $\approx p^M$ under independence of *error modes*—but **in practice** correlation $\rho > 0$ increases risk. **Divergent CoTs** reduce $\rho$ by decorrelating reasoning traces while preserving overlap on verifiable subtasks.

---


In [ ]:
# --- Divergent thought simulation (mock ChipLlama agents with different CoT biases) ---

@dataclass
class AgentProfile:
    name: str
    cot_bias: str  # narrative bias
    temperature: float


def simulate_plan(agent: AgentProfile, requirement: str) -> Dict:
    '''Produce a discrete plan vector in R^3: [arch_explore, constraint_tight, verify_depth].'''
    # Different biases shift the mean of a logistic-normal draw
    bias_map = {
        "formal-first": np.array([0.2, 0.9, 1.0]),
        "ppa-first": np.array([1.0, 0.4, 0.3]),
        "diversity-sampler": np.array([0.7, 0.5, 0.8]),
        "lint-conservative": np.array([0.3, 1.0, 0.9]),
    }
    mu = bias_map.get(agent.cot_bias, np.ones(3) * 0.5)
    noise = RNG.normal(scale=agent.temperature, size=3)
    logits = mu + noise
    x = 1 / (1 + np.exp(-logits))
    return {
        "agent": agent.name,
        "bias": agent.cot_bias,
        "vector": x,
        "summary": f"{agent.name} emphasizes {agent.cot_bias}",
    }


agents = [
    AgentProfile("Agent-α", "formal-first", 0.15),
    AgentProfile("Agent-β", "ppa-first", 0.18),
    AgentProfile("Agent-γ", "diversity-sampler", 0.22),
    AgentProfile("Agent-δ", "lint-conservative", 0.12),
]

requirement = "Close timing on a dual-clock FIFO macro with metastability-safe synchronizers."

plans = [simulate_plan(a, requirement) for a in agents]
X = np.stack([p["vector"] for p in plans])
mean_vec = X.mean(axis=0)
# Disagreement score: mean pairwise L2 distance
pairs = []
for i in range(len(plans)):
    for j in range(i + 1, len(plans)):
        pairs.append(np.linalg.norm(X[i] - X[j]))
disagreement = float(np.mean(pairs))

print("Requirement:", requirement)
print("Group disagreement Δ ≈", round(disagreement, 4))
for p in plans:
    print(p["summary"], "→", np.round(p["vector"], 3))

# Mock 'ground truth' good plan region
truth_center = np.array([0.55, 0.75, 0.85])
distances = [np.linalg.norm(v - truth_center) for v in X]
best_idx = int(np.argmin(distances))

# Single agent (ppa-first only) vs fusion by centroid
single = X[1]
centroid = mean_vec
ensemble_pick = X[best_idx]  # oracle for demo — in practice use votes on discrete actions

fig, ax = plt.subplots(figsize=(7, 5))
labels = ["arch_explore", "constraint_tight", "verify_depth"]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
single_r = np.concatenate([single, single[:1]])
centroid_r = np.concatenate([centroid, centroid[:1]])
angles_c = np.concatenate([angles, angles[:1]])
ax = plt.subplot(111, polar=True)
ax.plot(angles_c, single_r, color="#f85149", linewidth=2, label="Single agent (β)")
ax.fill(angles_c, single_r, color="#f85149", alpha=0.15)
ax.plot(angles_c, centroid_r, color="#3fb950", linewidth=2, label="Ensemble mean")
ax.fill(angles_c, centroid_r, color="#3fb950", alpha=0.12)
ax.set_thetagrids(angles * 180 / np.pi, labels)
ax.set_title("Divergent planning profiles (mock)")
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()

print("Closest-to-truth agent:", plans[best_idx]["agent"], "distance", round(distances[best_idx], 3))


## 4.4 ACE-RTL: Agentic Context Evolution

### 4.4.1 Iterated design as belief updating

ACE-RTL treats each iteration $t$ as producing RTL patch $\Delta_t$ and accumulating **context** $C_t$ (spec snippets, failing traces, coverage holes, lint waivers). Abstractly,
$$
C_{t+1} = \texttt{Merge}(C_t, \mathcal{O}_t), \quad \mathcal{O}_t = \text{Observations from verification},
$$
and the policy becomes
$$
\pi_\theta(\Delta_{t+1} \mid C_{t+1}) \quad \text{instead of} \quad \pi_\theta(\Delta \mid r \text{ alone}).
$$

### 4.4.2 Information-theoretic view

Let $H(\text{Bug})$ be entropy over fault localization. Good context updates obey approximate **information gain**:
$$
\mathbb{E}\big[ H(\text{Bug} \mid C_t) - H(\text{Bug} \mid C_{t+1}) \big] > 0.
$$

### 4.4.3 RTL-specialized LLMs

Domain adaptation (continued pretraining + SFT on RTL corpora + tool transcripts) shifts the tokenizer-induced inductive bias toward **syntactic validity** and **idiomatic concurrency patterns** (always_ff, clocking blocks, SVA templates). Formally, fine-tuning minimizes $\mathcal{L}_{\text{RTL}}$ on a mixture of next-token prediction and constrained decoding objectives.

---


In [ ]:
# --- ACE-RTL context evolution simulator ---

def simulate_ace_iterations(n: int = 8) -> pd.DataFrame:
    rows = []
    # latent unknown bug location in [0,1]
    bug = float(RNG.random())
    belief_mu = 0.5
    belief_sigma = 0.35
    context_tokens = 120
    for t in range(n):
        # policy proposes patch focus point
        proposal = float(RNG.normal(belief_mu, 0.12))
        proposal = min(max(proposal, 0.0), 1.0)
        # observation: noisy indicator if proposal overlaps bug region
        hit = abs(proposal - bug) < 0.08
        obs_signal = 1.0 if hit else 0.0
        # Bayesian-ish update on mu (toy)
        lr = 0.45
        belief_mu = (1 - lr) * belief_mu + lr * (bug if hit else proposal)
        belief_sigma *= 0.88 if hit else 0.95
        context_tokens += int(35 + 40 * obs_signal)
        rows.append({
            "iteration": t,
            "belief_mu": belief_mu,
            "belief_sigma": belief_sigma,
            "hit": hit,
            "context_tokens": context_tokens,
        })
    return pd.DataFrame(rows)


df_ace = simulate_ace_iterations(10)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(df_ace["iteration"], df_ace["belief_mu"], color=ACCENT, marker="o", label="Belief mean $\\hat\\mu_t$")
ax1.fill_between(
    df_ace["iteration"],
    df_ace["belief_mu"] - df_ace["belief_sigma"],
    df_ace["belief_mu"] + df_ace["belief_sigma"],
    color=ACCENT,
    alpha=0.15,
    label="$\pm\hat\sigma_t$",
)
ax1.set_xlabel("RTL iteration $t$")
ax1.set_ylabel("Latent bug location belief")
ax2 = ax1.twinx()
ax2.bar(df_ace["iteration"], df_ace["context_tokens"], color="#3fb950", alpha=0.25, width=0.45, label="Context size")
ax1.set_title("ACE-RTL: context growth vs. shrinking uncertainty (toy)")
ax1.grid(True)
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper left")
plt.tight_layout()
plt.show()

df_ace


## 4.5 Comparative Analysis: Axes, Metrics, and Visual Evidence

We compare systems along six interpretable axes (each in $[0,1]$ after min–max normalization for plotting):

1. **Verifiability** — strength of automated proof / compile / sim feedback in the loop.
2. **On-prem / IP** — ability to run without third-party inference on raw RTL.
3. **Multi-agent diversity** — intentional policy variance (CoT ensembles, role separation).
4. **Script agility** — ease of emitting tool glue (Python/Tcl) vs. raw RTL.
5. **Sample efficiency** — data hunger for fine-tuning (lower is better; inverted for radar).
6. **Interpretability** — auditability of traces $(r,z,a,o)$ or $(C_t,\Delta_t)$.

*Values below are illustrative mock scores consistent with architectural priors—not benchmark claims.*

---


In [ ]:
# --- Normalized radar (matplotlib) + table ---

systems = ["ChatEDA/AutoMage", "RTLCoder", "EDAid", "ACE-RTL"]
axes_labels = ["Verifiability", "On-prem IP", "Multi-agent", "Script agility", "Sample efficiency", "Interpretability"]

# Mock scores in [0,1]
scores = np.array([
    [0.55, 0.45, 0.35, 0.95, 0.40, 0.80],  # ChatEDA: great scripting, cloud-ish
    [0.90, 0.95, 0.25, 0.35, 0.70, 0.65],  # RTLCoder: verifier-strong, local
    [0.60, 0.55, 0.92, 0.50, 0.45, 0.55],  # EDAid: diversity
    [0.75, 0.60, 0.55, 0.50, 0.55, 0.75],  # ACE-RTL: context evolution
])

tab = pd.DataFrame(scores, index=systems, columns=axes_labels)
print(tab.round(2))

# Radar chart — matplotlib
N = len(axes_labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
angles = np.concatenate([angles, angles[:1]])

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(angles[:-1] * 180 / np.pi, axes_labels)
colors = ["#58a6ff", "#3fb950", "#d29922", "#a371f7"]
for row, name, c in zip(scores, systems, colors):
    values = np.concatenate([row, row[:1]])
    ax.plot(angles, values, color=c, linewidth=2, label=name)
    ax.fill(angles, values, color=c, alpha=0.10)
ax.set_ylim(0, 1)
ax.set_title("Architectural comparison (mock normalized scores)", y=1.08, fontsize=13)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.05))
plt.tight_layout()
plt.show()


In [ ]:
# --- Interactive Plotly radar (plotly_dark) ---

def radar_plotly(system: str, values: List[float]) -> go.Figure:
    theta = axes_labels + [axes_labels[0]]
    r = list(values) + [values[0]]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=r, theta=theta, fill="toself", name=system))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1], gridcolor="#30363d")),
        showlegend=True,
        title=f"Interactive radar — {system}",
        template="plotly_dark",
        paper_bgcolor=DARK_BG,
        plot_bgcolor=DARK_BG,
        font=dict(color="#c9d1d9"),
        margin=dict(t=60, b=40, l=50, r=50),
    )
    return fig

# Example: toggle which system to inspect
idx = 2  # EDAid
fig = radar_plotly(systems[idx], scores[idx].tolist())
fig.show()

# Multi-trace overlay
colors = ["#58a6ff", "#3fb950", "#d29922", "#a371f7"]
fig2 = go.Figure()
theta = axes_labels + [axes_labels[0]]
for row, name, c in zip(scores, systems, colors):
    r = list(row) + [row[0]]
    fig2.add_trace(go.Scatterpolar(r=r, theta=theta, mode="lines", name=name, line=dict(color=c, width=2)))
fig2.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Overlay radar — all systems (mock)",
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
)
fig2.show()


## 4.6 Synthesis: Choosing a Stack

**When ChatEDA/AutoMage shines:** heterogeneous toolchains, heavy Tcl/Python orchestration, and traceable decomposition for sign-off audits.

**When RTLCoder shines:** RTL-heavy flows with an open verification loop (Icarus, Verilator-class tools) and stringent IP containment.

**When EDAid shines:** planning under ambiguity—early microarchitecture exploration where multi-modal errors dominate.

**When ACE-RTL shines:** long-horizon debug/refinement where *context accumulation* is the bottleneck, not one-shot codegen.

### Limitations (intellectual honesty)

- Mock scores are **pedagogical**, not empirical PPA rankings.
- Real deployments couple these ideas: AutoMage-like controllers *over* RTLCoder backbones, ACE-style memory *with* divergent voting, etc.

### Further reading (representative themes)

- Self-instruction / weak-to-strong supervision for tool-use LLMs.
- RL from verifiable rewards (compiler feedback) for code models.
- Self-consistency and diversity in multi-agent reasoning.
- Iterated retrieval + memory for long-context RTL repair.

---


In [ ]:
# --- Optional: export mock comparison to JSON (for reproducible figures) ---
export = {
    "systems": systems,
    "axes": axes_labels,
    "scores": scores.tolist(),
    "notes": "Illustrative normalized scores for teaching; not measured benchmarks.",
}
print(json.dumps(export, indent=2)[:1200], "\n...")
